# Agentic Biomarker Discovery — Notebook 5
## Conversational Agent — Hybrid FAISS + Knowledge Graph

### Overview

Multi-turn conversational agent with **hybrid retrieval** combining FAISS (NB1)
and the Biomedical Knowledge Graph (NB4).

| Tool | Method | When to use |
|------|--------|-------------|
| `search()` | FAISS semantic similarity | Background, experimental results |
| `graph_search()` | KG co-occurrence traversal | Relationships, indirect associations |

**Key feature:** Subject-aware query reformulation prevents context poisoning across turns.
Demonstrated on an eight-question clinical narrative spanning miRNA, EGFR, BRCA1/2, and 
cross-cancer biomarker discovery.


---
## Cell 1 — Install Dependencies

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers bitsandbytes accelerate tqdm
print('✓ Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.4 MB/s eta 0:00:00
✓ Done


---
## Cell 2 — Mount Drive and Load All Artifacts

In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, pickle, re, warnings
import numpy as np
import networkx as nx
import faiss, torch
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

warnings.filterwarnings('ignore')
SEED   = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE_DIR  = '/content/drive/MyDrive/this project/Final'
INDEX_DIR = os.path.join(BASE_DIR, 'index')

# FAISS corpus
with open(os.path.join(INDEX_DIR, 'corpus_snippets.pkl'), 'rb') as f:
    corpus_snippets = pickle.load(f)
faiss_index = faiss.read_index(os.path.join(INDEX_DIR, 'bioasq_faiss.index'))
# Previous: 'sentence-transformers/all-MiniLM-L6-v2'
embed_model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO', device=device)

# Knowledge Graph (built in Notebook 4)
GRAPH_PATH = os.path.join(INDEX_DIR, 'biomedical_graph.pkl')
if os.path.exists(GRAPH_PATH):
    with open(GRAPH_PATH, 'rb') as f:
        G = pickle.load(f)
    print(f'OK Graph: {G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges')
else:
    G = None
    print('WARN Knowledge Graph not found — run Notebook 4 first')
    print('     graph_search() disabled; search() still works')

print(f'OK FAISS: {len(corpus_snippets):,} snippets | {faiss_index.ntotal:,} vectors')
print(f'   GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "CPU"}')


Mounted at /content/drive


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

OK Graph: 2,082 nodes | 8,173 edges
OK FAISS: 38,939 snippets | 38,939 vectors
   GPU: NVIDIA A100-SXM4-40GB


---
## Cell 3 — Load Llama 3.1-8B (4-bit)

In [3]:
HF_TOKEN = userdata.get('HF_TOKEN')

# ── Model Selection ────────────────────────────────────────────────
# USE_70B = True  → Llama 3.3 70B via Together.ai (better faithfulness,
#                   higher hop rate, requires TOGETHER_API_KEY secret)
# USE_70B = False → Llama 3.1 8B local 4-bit (faster load, lower VRAM)
USE_70B = False   # ← set True to use 70B via Together.ai

if USE_70B:
    # Together.ai inference — same model used for Condition D in NB2
    from openai import OpenAI
    together_client = OpenAI(
        api_key=userdata.get('TOGETHER_API_KEY'),
        base_url='https://api.together.xyz/v1',
    )
    MODEL_ID = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'
    tokenizer    = None
    model        = None
    llm_pipeline = None
    print(f'Using 70B via Together.ai: {MODEL_ID}')
    print('  Higher faithfulness, better format adherence, no local VRAM needed')
else:
    # Local 4-bit quantized Llama 3.1 8B
    MODEL_ID = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    )
    print(f'Loading {MODEL_ID} (4-bit local)...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map='auto', token=HF_TOKEN
    )
    model.eval()
    llm_pipeline = pipeline(
        'text-generation', model=model, tokenizer=tokenizer,
        max_new_tokens=600, temperature=0.2, do_sample=True, return_full_text=False,
    )
    print(f'Model ready | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

import transformers, warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings('ignore')
print('Deprecation warnings suppressed')


Loading meta-llama/Meta-Llama-3.1-8B-Instruct (4-bit local)...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model ready | VRAM: 6.1 GB
Deprecation warnings suppressed


---
## Cell 4 — Core Agent Functions

Three components:
- `search()` — retrieves top-k PubMed snippets from FAISS
- `call_llm()` — generates response from full conversation history
- `chat()` — the main conversational turn: retrieves evidence, builds context-aware prompt, updates history

In [4]:
# CRITICAL GROUNDING System Prompt
SYSTEM_PROMPT = (
    'You are an evidence-based biomedical research assistant specializing in '
    'biomarker discovery for pharmaceutical and life sciences applications.\n\n'
    'You have two retrieval tools:\n'
    '  search()       - Semantic similarity over PubMed.\n'
    '  graph_search() - Knowledge Graph traversal for relationships.\n\n'
    'CRITICAL GROUNDING RULE: You are an evidence-ONLY assistant. '
    'NEVER use internal training knowledge to provide a Final Answer. '
    'Your answer MUST be derived solely from the retrieved snippets. '
    'If the retrieved evidence does not contain the answer, you MUST either '
    'invoke search() again with a refined query, or state explicitly: '
    '"The answer is not found in the current PubMed evidence." '
    'Citing snippet numbers is mandatory for every factual claim.\n\n'
    'For each question:\n'
    '1. Read all retrieved evidence (marked FAISS or KGraph)\n'
    '2. Synthesize a grounded answer citing snippets by number\n'
    '3. Build on prior conversation when relevant\n'
    '4. Note any uncertainty or gaps'
)

# Cross-Encoder Reranker
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    RERANKER_AVAILABLE = True
    print('OK Cross-encoder reranker loaded')
except Exception as e:
    reranker = None
    RERANKER_AVAILABLE = False
    print(f'WARN Reranker unavailable: {e}')

ALPHA = 0.7  # semantic weight in hybrid scoring

BIO_SEEDS = {
    'egfr','brca1','brca2','tp53','kras','her2','vegf','alk','ret',
    'mirna','lncrna','mrna','alzheimer','cancer','tumor','carcinoma',
    'lymphoma','leukemia','melanoma','biomarker','mutation','expression',
    'pathway','receptor','kinase','inhibitor','protein','gene','genome',
    'methylation','apoptosis','metastasis','immunotherapy','chemotherapy',
}

def extract_entities(text: str) -> List[str]:
    if G is None: return []
    text_lower = text.lower()
    # Word-boundary regex prevents 'irs' matching 'theirs', 'ret' matching 'returned'
    graph_matches = [n for n in G.nodes()
                     if len(n) >= 3 and re.search(rf'\\b{re.escape(n)}\\b', text_lower)]
    seed_matches  = [s for s in BIO_SEEDS
                     if re.search(rf'\\b{re.escape(s)}\\b', text_lower)]
    caps_matches  = [w.lower() for w in re.findall(r'\b[A-Z]{2,8}\d*\b', text) if w.lower() in G]
    return sorted(set(graph_matches + seed_matches + caps_matches), key=len, reverse=True)[:5]

def search(query: str, k: int = 5, rerank_pool: int = 20) -> List[Dict]:
    """FAISS search with optional cross-encoder reranking."""
    vec = embed_model.encode([query], normalize_embeddings=True,
                              convert_to_numpy=True).astype(np.float32)
    pool_k = rerank_pool if RERANKER_AVAILABLE else k
    scores, indices = faiss_index.search(vec, pool_k)
    candidates = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        if idx < 0: continue
        s = corpus_snippets[idx]
        candidates.append({'rank': rank, 'score': float(score),
                            'text': s['text'], 'pubmed_url': s['pubmed_url'],
                            'source': 'FAISS'})
    if RERANKER_AVAILABLE and len(candidates) > k:
        pairs     = [(query, c['text'][:512]) for c in candidates]
        ce_scores = reranker.predict(pairs)
        for idx_c, c in enumerate(candidates):
            c['score'] = float(ce_scores[idx_c])
        candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)[:k]
        for rank, c in enumerate(candidates, 1): c['rank'] = rank
    return candidates[:k]

def graph_search(entity_name: str, top_n: int = 4,
                 history_entities: list = None) -> List[Dict]:
    """
    KG traversal with specificity-aware scoring.
    history_entities: entities from recent conversation — neighbors
    shared with these get a 1.5x boost to maintain subject-aware focus.
    """
    if G is None: return []
    query  = entity_name.lower().strip()
    target = query if query in G else next((n for n in G.nodes() if query in n), None)
    if not target: return []
    BETA = 0.5  # hub dampening
    history_set = set(history_entities or [])
    neighbors = []
    for nbr, dat in G[target].items():
        base  = dat['weight'] * np.log1p(G.degree(nbr))
        damp  = np.log1p(G.degree(nbr) + 1) ** BETA
        score = base / damp
        # Boost neighbors that appear in recent conversation history
        if nbr in history_set:
            score *= 1.5
        neighbors.append((nbr, dat['weight'], score))
    neighbors = sorted(neighbors, key=lambda x: x[2], reverse=True)[:top_n]
    results = []
    for rank, (name, weight, score) in enumerate(neighbors, 1):
        sids = G.nodes[name].get('snippet_ids', [])
        text = (corpus_snippets[sids[0]]['text']
                if sids and sids[0] < len(corpus_snippets)
                else f'[KGraph] {target} co-occurs with {name} (strength: {weight})')
        results.append({'rank': rank, 'score': float(score), 'text': text,
                         'pubmed_url': '', 'source': 'KGraph',
                         'entity': name, 'weight': weight})
    return results

def call_llm(messages: List[Dict]) -> str:
    """Generate response — uses Together.ai 70B or local 8B depending on USE_70B flag."""
    if USE_70B:
        resp = together_client.chat.completions.create(
            model=MODEL_ID, messages=messages,
            max_tokens=600, temperature=0.2,
        )
        return resp.choices[0].message.content.strip()
    else:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return llm_pipeline(
            prompt, max_new_tokens=600, temperature=0.2, do_sample=True
        )[0]['generated_text'].strip()

def reformulate_query(question: str, history: List[Dict]) -> str:
    """
    Subject-Aware Reformulation.
    Identifies the primary clinical subject from the USER own questions
    (not retrieved snippets) to prevent context poisoning from prior turns.
    """
    if not history: return question
    user_turns = [h['content'] for h in history if h['role'] == 'user'][-3:]
    prompt = (
        'Previous user questions (ignore retrieved snippets, user text only):\n'
        + '\n'.join(f'  - {q}' for q in user_turns)
        + f'\n\nFollow-up Question: {question}\n\n'
        'Step 1: Identify the PRIMARY CLINICAL SUBJECT the user explicitly named '
        '(a specific disease, gene, or biomarker from their own words).\n'
        'Step 2: Rewrite the follow-up as a standalone PubMed search query using '
        'that primary subject. Do NOT use entities from retrieved snippets.\n'
        'Return ONLY the final search query, nothing else.'
    )
    return call_llm([{'role': 'user', 'content': prompt}]).strip()

def chat(question: str, history: List[Dict], k: int = 5, verbose: bool = True) -> tuple:
    """
    Hybrid conversational turn:
    1. Subject-aware query reformulation (prevents context poisoning)
    2. FAISS retrieval + cross-encoder reranking
    3. KG entity traversal
    4. Weighted merge alpha=0.7 semantic / 0.3 graph
    5. Generate grounded answer (evidence-only)
    6. Update history
    """
    # 1. Reformulate
    search_query = reformulate_query(question, history)
    if verbose and search_query.lower() != question.lower():
        print(f'  [Reformulated: {search_query[:100]}]')

    # 2. FAISS + rerank
    faiss_results = search(search_query, k=k)
    if faiss_results:
        max_s = max(r['score'] for r in faiss_results)
        min_s = min(r['score'] for r in faiss_results)
        rng   = max_s - min_s if max_s != min_s else 1.0
        for r in faiss_results:
            r['hybrid_score'] = ALPHA * ((r['score'] - min_s) / rng)

    # 3. KG traversal
    graph_results, used_entity = [], None
    # Extract entities from recent history for context-aware graph boosting
    history_ents = []
    for h in history[-4:]:
        if h['role'] == 'user':
            history_ents.extend(extract_entities(h['content']))
    history_ents = list(set(history_ents))
    for ent in extract_entities(question):
        gr = graph_search(ent, top_n=3, history_entities=history_ents)
        if gr:
            graph_results = gr
            used_entity   = ent
            break
    if graph_results:
        max_g = max(r['score'] for r in graph_results)
        min_g = min(r['score'] for r in graph_results)
        rng_g = max_g - min_g if max_g != min_g else 1.0
        for r in graph_results:
            r['hybrid_score'] = (1 - ALPHA) * ((r['score'] - min_g) / rng_g)

    # 4. Weighted merge
    seen, merged = set(), []
    for s in faiss_results + graph_results:
        if s['text'] not in seen:
            seen.add(s['text'])
            merged.append(s)
    merged = sorted(merged, key=lambda x: x.get('hybrid_score', 0), reverse=True)
    for rank, s in enumerate(merged, 1): s['rank'] = rank

    context = '\n'.join([f"[{s['rank']}] ({s['source']}) {s['text']}" for s in merged])

    # 5. Generate grounded answer
    user_msg = (
        f'Retrieved Evidence (FAISS + Knowledge Graph):\n{context}\n\n'
        f'Question: {question}\n\n'
        f'Answer using ONLY the evidence above. Cite snippets by number. '
        f'If evidence is insufficient, state that explicitly. '
        f'Build on prior conversation when relevant.'
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages += history
    messages += [{'role': 'user', 'content': user_msg}]
    answer = call_llm(messages)

    # 6. Update history
    history = history + [
        {'role': 'user',      'content': question},
        {'role': 'assistant', 'content': answer},
    ]

    if verbose:
        print('\n' + '='*72)
        print(f'Q: {question}')
        if used_entity: print(f'  [KG entity: {used_entity}]')
        rerank_tag = ' + CrossEncoder' if RERANKER_AVAILABLE else ''
        print(f'\nEVIDENCE (FAISS{rerank_tag} + KGraph, alpha={ALPHA}):')
        for s in merged[:6]:
            hs = s.get('hybrid_score', 0)
            print(f"  [{s['rank']}] ({s['source']}, h={hs:.3f}) {s['text'][:140]}...")
        print(f'\nAGENT:\n{answer}')
        print('='*72)

    return answer, history, merged

print(f'OK Hybrid Agent: FAISS + CrossEncoder + KGraph')
print(f'   Reranker     : {RERANKER_AVAILABLE}')
print(f'   Hybrid alpha : {ALPHA} semantic / {1-ALPHA:.1f} graph')
print(f'   Graph        : {G is not None}')
print(f'   Grounding    : enforced (evidence-only)')
print(f'   Reformulation: subject-aware (prevents context poisoning)')


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

OK Cross-encoder reranker loaded
OK Hybrid Agent: FAISS + CrossEncoder + KGraph
   Reranker     : True
   Hybrid alpha : 0.7 semantic / 0.3 graph
   Graph        : True
   Grounding    : enforced (evidence-only)
   Reformulation: subject-aware (prevents context poisoning)


---
## Cell 5 — Guided Demo: Five-Question Clinical Narrative

These five questions build a coherent biomarker discovery story across multiple cancer types.
**Question 5 is the key test** — it references findings from Question 2 and can only be
answered correctly if the agent has access to the full conversation history.

| Turn | Question | What it tests |
|------|----------|---------------|
| 1 | miRNA in Alzheimer's | Domain-specific retrieval |
| 2 | EGFR mutation + NSCLC treatment | Gene-drug-biomarker linkage |
| 3 | Early vs late-stage colorectal cancer biomarkers | Multi-entity comparison |
| 4 | BRCA1/2 as biomarkers for PARP inhibitor response | Drug response prediction |
| 5 | Are any lung cancer biomarkers also relevant in colorectal cancer? | **Cross-turn reference — requires chat history** |

In [5]:
DEMO_QUESTIONS = [
    "Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?",
    "What is the relationship between EGFR mutations and biomarker-driven treatment selection in non-small cell lung cancer?",
    "Which biomarkers distinguish early-stage from late-stage colorectal cancer?",
    "How do BRCA1 and BRCA2 mutations serve as biomarkers for PARP inhibitor response in breast cancer?",
    "Based on what we just discussed about lung cancer biomarkers, are any of those also relevant in colorectal cancer?",
]

print('='*72)
print('GUIDED DEMO — Five-Question Biomarker Discovery Session')
print('='*72)
print('Question 5 is a cross-turn reference that requires chat history to answer correctly.\n')

# Initialize empty history
demo_history = []
demo_session = []  # stores (question, answer, snippets) for session summary

for i, question in enumerate(DEMO_QUESTIONS, 1):
    print(f'\n[TURN {i}/{len(DEMO_QUESTIONS)}]')
    answer, demo_history, snippets = chat(question, demo_history, k=7, verbose=True)
    demo_session.append({'turn': i, 'question': question, 'answer': answer, 'snippets': snippets})

print(f'\n✓ Demo session complete — {len(demo_history)//2} turns | History: {len(demo_history)} messages')

GUIDED DEMO — Five-Question Biomarker Discovery Session
Question 5 is a cross-turn reference that requires chat history to answer correctly.


[TURN 1/5]

Q: Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Blood miRNAs could be useful as biomarkers for exposure to nanoparticles. miR-298 regulates β-amyloid (Aβ) precursor protein-converting enzy...
  [2] (FAISS, h=0.229) The data indicate that micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represented in fetal hippo...
  [3] (FAISS, h=0.213) micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represented in fetal hippocampus, are differentia...
  [4] (FAISS, h=0.130) We previously studied microRNAs (miRNAs) in AD autopsy brain samples and reported a connection between miR-137, -181c, -9, -29a/b and AD, th...
  [5] (FAISS, h=0.04

---
## Cell 6 — Session Summary

After a session, the agent synthesizes key biomarker findings across all turns
into a structured clinical summary — demonstrating the value of accumulated context.

In [6]:
def session_summary(history: List[Dict]) -> str:
    """Ask the agent to synthesize all findings from the session."""
    summary_prompt = (
        "Based on our entire conversation above, please provide a structured clinical summary of "
        "the key biomarker findings we discussed. Organize by disease area and include:\n"
        "1. Key biomarkers identified per disease\n"
        "2. Their clinical utility (diagnostic, prognostic, or predictive)\n"
        "3. Any cross-disease patterns or overlaps you identified\n"
        "4. Gaps in the evidence that warrant further research"
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages += history
    messages += [{'role': 'user', 'content': summary_prompt}]

    summary = call_llm(messages)
    return summary

print('Generating session summary...')
summary = session_summary(demo_history)

print('\n' + '='*72)
print('SESSION SUMMARY — Key Biomarker Findings')
print('='*72)
print(summary)
print('='*72)

# Save summary to Drive
summary_path = os.path.join(BASE_DIR, 'session_summary.txt')
with open(summary_path, 'w') as f:
    f.write('BIOMARKER DISCOVERY SESSION SUMMARY\n')
    f.write('='*60 + '\n\n')
    for entry in demo_session:
        f.write(f"Q{entry['turn']}: {entry['question']}\n")
        f.write(f"A: {entry['answer']}\n\n")
    f.write('\nSYNTHESIS:\n' + summary)
print(f'\n✓ Session saved → {summary_path}')

Generating session summary...

SESSION SUMMARY — Key Biomarker Findings
**Clinical Summary of Key Biomarker Findings**

**Disease Area: Lung Cancer**

1. **Key Biomarkers Identified:**
	* EGFR mutations (exon 19 deletions, L858R substitution)
	* EGFR expression (immunohistochemistry)
2. **Clinical Utility:**
	* Predictive biomarkers for response to EGFR tyrosine kinase inhibitors (TKIs) in non-small cell lung cancer (NSCLC)
	* Diagnostic biomarkers for identifying patients with EGFR mutations
3. **Cross-Disease Patterns or Overlaps:**
	* EGFR mutations and EGFR expression are also relevant in colorectal cancer, particularly in predicting response to EGFR-targeted therapies
4. **Gaps in the Evidence:**
	* Specific biomarkers associated with EGFR TKI response in colorectal cancer
	* Further research is needed to identify additional biomarkers and their relationship with EGFR TKI response in NSCLC and colorectal cancer

**Disease Area: Colorectal Cancer**

1. **Key Biomarkers Identified:*

---
## Cell 7 — Interactive Conversational Mode with Chat History

**Commands:**
- Type any biomedical question and press Enter
- Type `history` to see the current conversation history
- Type `summary` to get a synthesis of the session so far
- Type `reset` to start a fresh session
- Type `quit` to exit

This cell is an open-ended conversational interface. The agent remembers everything within the session. It demonstrates the agent in open-ended chat mode using Python's `input()` prompt — the same underlying architecture as Cell 5 and Cell 7b but allowing arbitrary user queries across as many turns as desired.

**Not executed in this notebook** — `input()` blocks Jupyter execution and
is not suitable for automated recording. To use interactively, run this
cell in an active Colab session and type questions at the prompt.

This cell shows that the hybrid FAISS+KG agent can be extended to a
full conversational interface or embedded in a web application (e.g.
Streamlit, Gradio) with minimal additional code.

In [ ]:
free_history = []
free_session = []
turn = 0

print('='*60)
print('  BIOMARKER DISCOVERY AGENT — Conversational Mode')
print('  Powered by Llama-3.1-8B + BioASQ PubMed Corpus')
print('='*60)
print('Commands: history | summary | reset | quit\n')

while True:
    try:
        user_input = input(f'[Turn {turn+1}] Your question: ').strip()
    except EOFError:
        break

    if not user_input:
        continue

    if user_input.lower() == 'quit':
        print(f'\nSession ended. Total turns: {turn}')
        break

    elif user_input.lower() == 'reset':
        free_history = []
        free_session = []
        turn = 0
        print('\n✓ Session reset — starting fresh\n')
        continue

    elif user_input.lower() == 'history':
        if not free_history:
            print('  No history yet.\n')
        else:
            print('\n--- CONVERSATION HISTORY ---')
            for j in range(0, len(free_history), 2):
                q = free_history[j]['content']
                a = free_history[j+1]['content'][:200] + '...' if len(free_history[j+1]['content']) > 200 else free_history[j+1]['content']
                print(f"Q{j//2+1}: {q}")
                print(f"A: {a}\n")
        continue

    elif user_input.lower() == 'summary':
        if not free_history:
            print('  No conversation to summarize yet.\n')
        else:
            print('\nGenerating summary...')
            summ = session_summary(free_history)
            print('\n--- SESSION SUMMARY ---')
            print(summ)
            print()
        continue

    else:
        turn += 1
        answer, free_history, snippets = chat(user_input, free_history, k=7, verbose=True)
        free_session.append({'turn': turn, 'question': user_input, 'answer': answer})
        print(f'  [History: {len(free_history)//2} turns in memory]\n')

  BIOMARKER DISCOVERY AGENT — Conversational Mode
  Powered by Llama-3.1-8B + BioASQ PubMed Corpus
Commands: history | summary | reset | quit

[Turn 1] Your question: Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Blood miRNAs could be useful as biomarkers for exposure to nanoparticles. miR-298 regulates β-amyloid (Aβ) precursor protein-converting enzy...
  [2] (FAISS, h=0.229) The data indicate that micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represented in fetal hippo...
  [3] (FAISS, h=0.213) micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represented in fetal hippocampus, are differentia...
  [4] (FAISS, h=0.130) We previously studied microRNAs (miRNAs) in AD autopsy brain samples and reported a connection between miR-137, -181c, -9, -29a/b and AD, th...
  [5] (FAISS, h=0.049) We previously observed that miR-137, -181c, -9, and 29a/b post-transcriptionally regulate SPT levels, and the corresponding miRNA levels in ...
  [6] (

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Reformulated: EGFR mutations]

Q: What is the relationship between EGFR mutations and treatment selection in non-small cell lung cancer?
  [KG entity: egfr]

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Mutations of the epidermal growth factor receptor (EGFR) gene have been reported in non-small-cell lung cancer (NSCLC), especially in patien...
  [2] (KGraph, h=0.300) Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the recep...
  [3] (KGraph, h=0.233) Upregulation of microRNA-203 is associated with advanced tumor progression and poor prognosis in epithelial ovarian cancer...
  [4] (FAISS, h=0.217) Epidermal growth factor receptor (EGFR) gene mutations have been found in a subset of non-small cell lung cancer (NSCLC) with good clinical ...
  [5] (FAISS, h=0.203) Two types of epidermal growth factor receptor (EGFR) mutations in exon 19 and exon 21 (ex19del and L858R) are p

...

  [Reformulated: pancreatic cancer]

Q: Which protein biomarkers are shared between early-stage pancreatic cancer and hepatocellular carcinoma?

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Pancreatic neuroendocrine tumors (PNETs) are a characteristic feature of the tumor syndromes multiple endocrine neoplasia type 1 (MEN-1) and...
  [2] (FAISS, h=0.453) Pancreatic endocrine tumors occur sporadically and as part of the multiple endocrine neoplasia type 1 (MEN 1) and von Hippel-Lindau (VHL) sy...
  [3] (FAISS, h=0.289) Mitotic disruption and reduced clonogenicity of pancreatic cancer cells in vitro and in vivo by tumor treating fields....
  [4] (FAISS, h=0.284) In pancreatic cancer xenografts obtained directly from patients with pancreas cancer, the agent resulted in a marked suppression of tumor gr...
  [5] (FAISS, h=0.101) CD44v2 and CD44v6 may be useful markers for poor prognosis in curatively resected primary pancreatic cancer...
  [6] (FAISS, h=0.019)

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Reformulated: triple-negative breast cancer]

Q: What genes are co-expressed with TP53 in triple-negative breast cancer and could serve as companion biomarkers?
  [KG entity: tp53]

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Triple-negative breast cancers (TNBC) do not represent a single disease subgroup and are often aggressive breast cancers with poor prognoses...
  [2] (FAISS, h=0.590) Proteomic profiling of triple-negative breast carcinomas in combination with a three-tier orthogonal technology approach identifies Mage-A4 ...
  [3] (FAISS, h=0.528) As seen in this case, most breast cancers in patients with LFS exhibit a triple-positive phenotype (estrogen receptor-positive/progesterone ...
  [4] (FAISS, h=0.510) We have previously reported an array comparative genomic hybridization profile that identifies triple-negative breast cancers (TNBC), with B...
  [5] (KGraph, h=0.300) Coding sequence mutations in e.g. RET, GDNF, EDNRB, EDN3, and SOX10 le

Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Reformulated: TP53]

Q: Based on everything we have discussed, which single biomarker appears most frequently across multiple cancer types?

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) The tumor suppressor p53, encoded by the TP53 gene, is recognized as the guardian of the human genome because it regulates many downstream g...
  [2] (FAISS, h=0.478) The tumour suppressor gene TP53 is the most frequently mutated gene in cancer. Wild-type p53 can suppress tumour development by multiple pat...
  [3] (FAISS, h=0.153) TP53 aberrations (n = 3 patients) varied by type and location between primary and metastatic tumors sites but were intra-tumorally conserved...
  [4] (FAISS, h=0.116) Fluorescence in situ hybridization of TP53 for the detection of chromosome 17 abnormalities in myelodysplastic syndromes....
  [5] (FAISS, h=0.060) Germline TP53 mutations predispose to early onset breast cancer in women and are associated with Li-Fraumeni syndrome....
  [6] (FA

---
## Cell 7b — Pre-Run Biomarker Discovery Demo

Runs all 8 demonstration questions sequentially with no user input required.
Designed for video recording and animated GIF capture.
Each turn shows: reformulated query, evidence sources (FAISS/KGraph), and agent answer.

**Questions build a clinical narrative:**
- Turns 1-2: Establish disease subjects (Alzheimer miRNA, EGFR/NSCLC)
- Turn 3: Cross-turn reference (EGFR resistance mechanisms)
- Turn 4: New subject (BRCA1/2, PARP inhibitors)
- Turn 5: Cross-disease test (lung → colorectal) ← key chat history proof
- Turns 6-7: Multi-hop discovery questions
- Turn 8: Synthesis across all turns ← proves session memory

In [7]:
import warnings, transformers
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

DEMO_QUESTIONS_EXTENDED = [
    # Turn 1: Establish first subject — Alzheimer miRNA
    "Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?",
    # Turn 2: New subject — EGFR in lung cancer
    "What is the relationship between EGFR mutations and biomarker-driven treatment selection in non-small cell lung cancer?",
    # Turn 3: Follow-up requiring Turn 2 memory
    "Which of those EGFR-targeted therapies show resistance mechanisms involving secondary mutations?",
    # Turn 4: New subject — BRCA
    "How do BRCA1 and BRCA2 mutations serve as predictive biomarkers for PARP inhibitor response in breast cancer?",
    # Turn 5: Cross-turn reference — requires EGFR memory from Turn 2
    "Are any of the lung cancer biomarkers we discussed also relevant in colorectal cancer?",
    # Turn 6: Multi-hop discovery
    "Which protein biomarkers are shared between pancreatic cancer and hepatocellular carcinoma?",
    # Turn 7: TP53 discovery
    "What genes are co-expressed with TP53 in triple-negative breast cancer and could serve as companion biomarkers?",
    # Turn 8: Full session synthesis
    "Based on everything we have discussed, which single biomarker appears most frequently across multiple cancer types?",
]

SEP = '=' * 72

print(SEP)
print('  BIOMARKER DISCOVERY AGENT — Pre-Run Clinical Demo')
print('  Llama-3.1-8B + S-PubMedBert-MS-MARCO + BioASQ Knowledge Graph')
print('  Hybrid FAISS + CrossEncoder + Knowledge Graph Retrieval')
print(SEP)
print(f'  {len(DEMO_QUESTIONS_EXTENDED)} questions | Chat history active across all turns')
print(SEP)

demo_history  = []
demo_session  = []

for turn_num, question in enumerate(DEMO_QUESTIONS_EXTENDED, 1):
    print(f'\n{"─"*72}')
    print(f'TURN {turn_num} of {len(DEMO_QUESTIONS_EXTENDED)}')
    print(f'{"─"*72}')
    answer, demo_history, snippets = chat(
        question, demo_history, k=5, verbose=True
    )
    demo_session.append({
        'turn': turn_num, 'question': question,
        'answer': answer, 'snippets': snippets,
    })
    print(f'\n[Session memory: {len(demo_history)//2} turns active]')

print(f'\n{SEP}')
print(f'Demo complete — {len(demo_history)//2} turns | {len(demo_session)} questions')
print(SEP)
print('\nGenerating clinical session summary...')

summary = session_summary(demo_history)
print('\n' + SEP)
print('SESSION SUMMARY — Key Biomarker Findings Across All Turns')
print(SEP)
print(summary)
print(SEP)

import os
summary_path = os.path.join(BASE_DIR, 'demo_session_summary.txt')
with open(summary_path, 'w') as f:
    f.write('BIOMARKER DISCOVERY AGENT — Demo Session Summary\n')
    f.write('='*60 + '\n\n')
    for entry in demo_session:
        f.write(f"Q{entry['turn']}: {entry['question']}\n")
        f.write(f"A: {entry['answer']}\n\n")
    f.write('\nSYNTHESIS:\n' + summary)
print(f'\nSaved -> {summary_path}')


  BIOMARKER DISCOVERY AGENT — Pre-Run Clinical Demo
  Llama-3.1-8B + S-PubMedBert-MS-MARCO + BioASQ Knowledge Graph
  Hybrid FAISS + CrossEncoder + Knowledge Graph Retrieval
  8 questions | Chat history active across all turns

────────────────────────────────────────────────────────────────────────
TURN 1 of 8
────────────────────────────────────────────────────────────────────────

Q: Which miRNA biomarkers are downregulated in Alzheimer's disease and what pathways do they regulate?

EVIDENCE (FAISS + CrossEncoder + KGraph, alpha=0.7):
  [1] (FAISS, h=0.700) Blood miRNAs could be useful as biomarkers for exposure to nanoparticles. miR-298 regulates β-amyloid (Aβ) precursor protein-converting enzy...
  [2] (FAISS, h=0.193) The data indicate that micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represented in fetal hippo...
  [3] (FAISS, h=0.177) micro-RNAs encoding miR-9, miR-124a, miR-125b, miR-128, miR-132 and miR-219 are abundantly represen